# slv_motivosreprovacao

In [16]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType

print(f"{'='*80}")
print("CONSTRUÇÃO DA TABELA SILVER: slv.motivosreprovacao")
print(f"{'='*80}\n")


# PARAMETERS

# Default. Quando correr pelo pipeline, este valor é substituído.
run_id = "manual"

print("Parâmetros recebidos pelo notebook:")
print("run_id:", run_id)


# 1. CARREGAR DADOS BRONZE (E AUXILIARES)
df_brz = spark.read.table("brz.motivosreprovacao")
df_corr = spark.read.table("brz.correspondencias_final")
df_siss = spark.read.table("brz.exploracoes_siss")
df_revistos = spark.read.table("brz.motivosrevistos")
df_sindromes = spark.read.table("brz.motivossindromes")


#2. DEDUPLICAÇÃO E LIMPEZA BASE REPROVAÇÕES
#Selecionar snapshot mais recente por ID

win_snap = Window.partitionBy("id").orderBy(F.desc("meta_source_file_date"), F.desc("audit_brz_load_timestamp"))

df_base = df_brz.withColumn("row_rank", F.row_number().over(win_snap)) \
    .filter(F.col("row_rank") == 1) \
    .drop("row_rank")

df_base = df_base.select(
    F.col("id").cast(IntegerType()).alias("id_reprovacao"),
    F.trim(F.col("ncv")).alias("ncv"),
    F.to_date(F.col("datacontrolo")).alias("data_controlo"),
    F.col("numanimais").cast(IntegerType()).alias("qtd_animais_reprovados"),
    F.trim(F.col("especie")).alias("especie"),
    F.trim(F.col("motivorejeicao")).alias("motivo_original"),
    F.trim(F.col("codigoexploracaoorigem")).alias("codigo_exploracao_origem"),
    F.col("departamento").alias("regiao_id"),
    F.trim(F.col("nome")).alias("regiao"),
    F.trim(F.col("nome2")).alias("matadouro"),
    F.lit("PORTUGAL").alias("pais_de_origem_original"), # Para ajudar no fallback depois
    "meta_source_file", "meta_source_file_date"
).filter(
    F.upper(F.col("especie")) == "SUÍNOS"
).filter(
    ~F.upper(F.col("codigo_exploracao_origem")).isin("NAN", "NULL", "")
)


#3. LIMPEZA DE CÓDIGO (REGEX + CASE WHEN) - ESPELHO DA TRICHINELLA
df_transform = df_base.withColumn(
    "marca_temp",
    F.trim(F.regexp_replace(F.col("codigo_exploracao_origem"), "(?i)(CANCELADO|ANULADA|INATIVA|INATIVO|SIPACE)", ""))
)

df_transform = df_transform.withColumn(
    "codigo_reduzida_temp",
    F.when(F.col("marca_temp").startswith("PT"), F.substring(F.col("marca_temp"), 3, 100))
     .otherwise(F.col("marca_temp"))
)

df_transform = df_transform.withColumn(
    "marca_temp",
    F.when(F.col("marca_temp") == "PTRB4C0PTRB66G", "PTRB4C0")
     .when(F.col("marca_temp") == "PTWF", "PTWF04B")
     .when(F.col("marca_temp") == "PTAV08", "PTAVA08")
     .when(F.col("marca_temp") == "RG5G8A", "RG5G8")
     .when(F.col("marca_temp") == "PTEAR49SS", "PTEAR49")
     .when(F.col("marca_temp") == "PTBX70", "PTBJX70")
     .when(F.col("marca_temp") == "PTA22T", "PTJA22T")
     .when(F.col("marca_temp") == "V50G", "PTVN50G")
     .when(F.col("marca_temp") == "PRRGV06", "PTRGV06")
     .when(F.col("marca_temp") == "9800953", "PT9U08V")
     .when(F.col("marca_temp") == "PYJAX74", "PTJAX74")
     .when(F.col("marca_temp") == "G24112011YD37C", "PTYD37C")
     .when(F.col("marca_temp") == "TF1Z4", "PTTF1Z4A")
     .when(F.col("marca_temp") == "KG1R6", "PTKG1R6A")
     .when(F.col("marca_temp") == "RS3F1", "PTRS3F1A")
     .when(F.col("marca_temp") == "RB04M", "PTRB04MA")
     .when(F.col("marca_temp").isin("PTRB8G5A"), "PTRB8G5")
     .when(F.col("marca_temp") == "PTRB5G2A", "PTRB5G2")
     .when(F.col("marca_temp") == "PTHE20TA", "PTHE20T")
     .when(F.col("marca_temp") == "VS67C", "PTVS67CA")
     .when(F.col("marca_temp") == "RWP48", "PTRWP48G")
     .when(F.col("marca_temp").isin("RY69T", "PTRY69TV"), "PTRY69T")
     .when(F.col("marca_temp") == "VS0AD", "PTVS0ADA")
     .when(F.col("marca_temp").isin("RY41H", "PTRY41HV"), "PTRY41H")
     .when(F.col("marca_temp") == "AAFG5", "PTAAFG58")
     .when(F.col("marca_temp").isin("RY45B", "PTRY45BV"), "PTRY45B")
     .when(F.col("marca_temp") == "R53F1", "PTR53F1A")
     .when(F.col("marca_temp") == "RG01S", "PTRG01SI")
     .when(F.col("marca_temp") == "RB542", "PTRB542A")
     .when(F.col("marca_temp") == "RG5G3", "PTRG5G3A")
     .when(F.col("marca_temp") == "HM02Z", "PTHM02ZA")
     .when(F.col("marca_temp") == "RB5G2", "PTRB5G2")
     .when(F.col("marca_temp") == "ES320820056601ES82OR402", "ES82OR402")
     .when(F.col("marca_temp").rlike(r"^\d{2}PT\d{5}$"), F.substring(F.col("marca_temp"), 3, 100))
     .when(F.col("marca_temp").startswith("PT") & (F.length(F.col("codigo_reduzida_temp")) == 6), F.substring(F.col("codigo_reduzida_temp"), 1, 5))
     .otherwise(F.col("marca_temp"))
)

# Join com a Tabela de Correspondências
df_corr_clean = df_corr.select(
    F.trim(F.col("codigoexploracaoorigem")).alias("join_key"),
    F.trim(F.col("sugestao")).alias("sugestao_val")
).distinct()

df_corr_clean = df_corr_clean.withColumn("sugestao_val", F.when(F.upper(F.col("sugestao_val")).isin("NAN", "NULL", ""), F.lit(None)).otherwise(F.col("sugestao_val")))

df_joined = df_transform.join(df_corr_clean, df_transform.marca_temp == df_corr_clean.join_key, "left")

df_pre_siss = df_joined.withColumn("marca_base", F.coalesce(F.col("sugestao_val"), F.col("marca_temp"))) \
    .withColumn("marca_base", F.when((F.upper(F.col("pais_de_origem_original")) == "PORTUGAL") & (F.length(F.col("marca_base")) == 5), F.concat(F.lit("PT"), F.col("marca_base"))).otherwise(F.col("marca_base"))) \
    .drop("sugestao_val", "join_key", "marca_temp", "codigo_reduzida_temp")


# 4. CRIAR TABELA DE CORRESPONDÊNCIA SISS (O MOTOR EM CASCATA)
df_keys = df_pre_siss.filter(F.col("marca_base").startswith("PT")).select("marca_base").distinct()
df_keys = df_keys.withColumn("red_tri", F.substring(F.col("marca_base"), 3, 100))
df_keys = df_keys.withColumn("par_tri", F.when(~F.substring(F.col("red_tri"), 1, 5).rlike(r"^\d+$"), F.substring(F.col("red_tri"), 1, 5)).otherwise(F.lit(None)))

win_siss = Window.partitionBy("marca").orderBy(F.desc("meta_source_file_date"))
df_siss_latest = df_siss.withColumn("rn", F.row_number().over(win_siss)).filter(F.col("rn") == 1)

df_oficial = df_siss_latest.select(F.trim(F.col("marca")).alias("marca_oficial"))
df_oficial = df_oficial.withColumn("red_ofi", F.when(F.col("marca_oficial").startswith("PT"), F.substring(F.col("marca_oficial"), 3, 100)).otherwise(F.col("marca_oficial")))
df_oficial = df_oficial.withColumn("par_ofi", F.when(~F.substring(F.col("red_ofi"), 1, 5).rlike(r"^\d+$"), F.substring(F.col("red_ofi"), 1, 5)).otherwise(F.lit(None)))

siss_exata = df_oficial.select("marca_oficial").dropDuplicates(["marca_oficial"])
siss_reduz = df_oficial.select("marca_oficial", "red_ofi").dropDuplicates(["red_ofi"])
siss_parci = df_oficial.filter(F.col("par_ofi").isNotNull()).select("marca_oficial", "par_ofi").dropDuplicates(["par_ofi"])

matched = df_keys.join(siss_exata, df_keys.marca_base == siss_exata.marca_oficial, "left").withColumnRenamed("marca_oficial", "m1")
matched = matched.join(siss_reduz, matched.red_tri == siss_reduz.red_ofi, "left").withColumnRenamed("marca_oficial", "m2").drop("red_ofi")
matched = matched.join(siss_parci, matched.par_tri == siss_parci.par_ofi, "left").withColumnRenamed("marca_oficial", "m3").drop("par_ofi")

matched = matched.withColumn("arr", F.array_distinct(F.expr("filter(array(m1, m2, m3), x -> x is not null)")))
df_bridge = matched.withColumn("marca_siss", F.array_join(F.col("arr"), "; ")) \
                   .withColumn("flg_siss_multi", F.when(F.size(F.col("arr")) > 1, "Sim").otherwise("Não")) \
                   .select("marca_base", "marca_siss", "flg_siss_multi")


# 5. SUBSTITUIÇÃO NA BASE PRINCIPAL E TRATAMENTO DE MOTIVOS (SÍNDROMES)
df_final = df_pre_siss.join(df_bridge, "marca_base", "left")

df_final = df_final.withColumn("marca", F.when(F.col("marca_siss").isNotNull() & (F.col("marca_siss") != ""), F.col("marca_siss")).otherwise(F.col("marca_base")))
df_final = df_final.withColumn("codigo_reduzida", F.when(F.substring(F.col("marca"), 1, 2) == "PT", F.substring(F.col("marca"), 3, 100)).otherwise(F.col("marca"))) \
                   .withColumn("pais_de_origem", F.when(F.col("marca").startswith("PT"), "Portugal").when(F.col("marca").rlike("^(ES|Es|es)"), "Espanha").otherwise(F.col("pais_de_origem_original"))) \
                   .filter(F.upper(F.col("pais_de_origem")) == "PORTUGAL") \
                   .drop("pais_de_origem_original", "marca_base", "marca_siss")

#--- TRATAMENTO DOS MOTIVOS ---
mapping_revisao = df_revistos.select(F.trim(F.col("motivo_inicial")).alias("ref_motivo_inicial"), F.trim(F.col("motivo_final")).alias("motivo_final_clean")).distinct()
mapping_categoria = df_sindromes.select(F.trim(F.col("motivorejeicao")).alias("ref_motivo_busca"), F.trim(F.col("categoriamotivo")).alias("categoriamotivo")).distinct()

# Juntar para substituir motivo original pela revisão
df_final = df_final.join(mapping_revisao, F.upper(df_final.motivo_original) == F.upper(mapping_revisao.ref_motivo_inicial), "left")
df_final = df_final.withColumn("motivo_rejeicao_final", F.coalesce(F.col("motivo_final_clean"), F.col("motivo_original")))

# Juntar para ir buscar a Categoria do Motivo (Síndrome)
df_final = df_final.join(mapping_categoria, F.upper(df_final.motivo_rejeicao_final) == F.upper(mapping_categoria.ref_motivo_busca), "left")

# 6. SELECÇÃO E AUDITORIA (SEM GROUP BY)
df_silver = df_final.select(
    "id_reprovacao",
    "ncv",
    "data_controlo",
    "regiao_id",
    "regiao",
    "matadouro",
    "especie",
    F.col("motivo_rejeicao_final").alias("motivo_rejeicao"),
    F.coalesce(F.col("categoriamotivo"), F.lit("SEM CATEGORIA")).alias("sindrome_rejeicao"),
    "qtd_animais_reprovados",
    "codigo_exploracao_origem",
    "marca",
    "codigo_reduzida",
    "pais_de_origem",
    "flg_siss_multi",
    "meta_source_file",
    "meta_source_file_date",
    F.current_timestamp().alias("audit_silver_refresh_timestamp"),
    F.lit("brz.motivosreprovacao").alias("audit_source_bronze"),
    F.lit(run_id).alias("audit_run_id")
)

# 7. GUARDAR NA SILVER
spark.sql("CREATE SCHEMA IF NOT EXISTS slv")

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("slv.motivosreprovacao")

print(f"Sucesso! Tabela slv.motivosreprovacao gerada com {df_silver.count()} linhas.")

print("Sucesso! Tabela slv.motivosreprovacao gerada perfeitamente alinhada com Trichinella.")

StatementMeta(, a4781435-5177-44a8-b28d-d77fd8611d1d, 3, Finished, Available, Finished, False)

CONSTRUÇÃO DA TABELA SILVER: slv.motivosreprovacao



Sucesso! Tabela slv.motivosreprovacao gerada com 40152 linhas.
Sucesso! Tabela slv.motivosreprovacao gerada perfeitamente alinhada com Trichinella.


#### Validaçao

In [17]:
# from pyspark.sql import functions as F

# # 1. Carregar a Tabela Silver
# table_name = "slv.motivosreprovacao"
# df_slv = spark.read.table(table_name)

# print(f"{'='*80}")
# print(f"RELATÓRIO DE VALIDAÇÃO SILVER: {table_name}")
# print(f"{'='*80}\n")

# # --- PASSO 1: Volume e Integridade ---
# total_rows = df_slv.count()
# print(f"1. ESTATÍSTICAS DE VOLUME")
# print(f"   - Total de registos de reprovação: {total_rows}")
# print(f"   - Total de animais reprovados (Soma): {df_slv.agg(F.sum('qtd_animais_reprovados')).collect()[0][0]}\n")

# # --- PASSO 2: Validação do Mapping (Trichinella Reference) ---
# print("2. EFICÁCIA DO MAPEAMENTO (JOIN COM TRICHINELLA)")
# # Calculamos quantos registos foram encontrados na mestre vs quantos ficaram como 'NA'
# mapping_stats = df_slv.groupBy("marca").agg(
#     F.count("*").alias("nr_registos"),
#     F.sum("qtd_animais_reprovados").alias("total_animais")
# # ).withColumn("status_match", F.when(F.upper(F.col("marca")) == "NA", "SEM CORRESPONDÊNCIA (NA)").otherwise("MAPEADO COM SUCESSO"))

# # mapping_summary = mapping_stats.groupBy("status_match").agg(
#     F.sum("nr_registos").alias("registos"),
#     F.sum("total_animais").alias("animais")
# )
# mapping_summary.show()

# # --- PASSO 3: Análise de códigos 'NA' (Para melhoria futura) ---
# print("3. TOP 10 CÓDIGOS ORIGINAIS NÃO MAPEADOS (PORQUE SÃO 'NA'?)")
# # Isto ajuda-te a ver se há marcas grandes nas reprovações que não existem na Trichinella
# df_slv.filter(F.upper(F.col("marca")) == "NA") \
#     .groupBy("codigo_exploracao_origem") \
#     .agg(F.sum("qtd_animais_reprovados").alias("animais_perdidos")) \
#     .orderBy(F.desc("animais_perdidos")) \
#     .show(10)

# # --- PASSO 4: Distribuição Geográfica e de Negócio ---
# print("4. TOP 10 MOTIVOS DE REJEIÇÃO")
# df_slv.groupBy("motivo_rejeicao") \
#     .agg(F.sum("qtd_animais_reprovados").alias("total")) \
#     .orderBy(F.desc("total")) \
#     .show(10, truncate=False)

# print("5. TOP 10 MATADOUROS POR VOLUME DE REPROVAÇÕES")
# df_slv.groupBy("matadouro", "ncv") \
#     .agg(F.sum("qtd_animais_reprovados").alias("total_reprovados")) \
#     .orderBy(F.desc("total_reprovados")) \
#     .show(10, truncate=False)

# # --- PASSO 5: Consistência Temporal ---
# print("6. DISTRIBUIÇÃO POR DATA DE CONTROLO (ANOS)")
# df_slv.withColumn("ano", F.year("data_controlo")) \
#     .groupBy("ano").count().orderBy("ano").show()

# # --- PASSO 6: Preview dos Dados ---
# print("7. AMOSTRA FINAL DOS DADOS (LIMPOS E MAPEADOS)")
# display(df_slv.orderBy(F.desc("data_controlo")))

# print(f"\n{'='*80}")
# print("VALIDAÇÃO CONCLUÍDA")
# print(f"{'='*80}")

StatementMeta(, 9a85d58f-8729-4af0-8910-7d08ddf22258, 21, Finished, Available, Finished, False)